# 4. Model — Random Forest

This notebook builds the second model of the project, a **random forest**, with the same protocol as the logistic-regression baseline (`1_BaselineModel.ipynb`): the same 7 temporal splits, the same recipe and the same feature-selection strategy, so the two models can be compared split by split.

- **Assessment:** inside each split, the recipe, the feature selection, the forest and the cut-off are fitted on the training rows only; the validation rows are only scored.
- **Cut-off:** a forest predicts its own training rows almost perfectly, so the cut-off is chosen on the **out-of-bag** probabilities (each row scored only by the trees that did not see it), not on the training probabilities used for the logistic regression.
- **Against the baseline:** paired comparison with the logistic regression on the same splits.
- **Tuning:** a random search of the forest's hyperparameters, scored on splits that took no part in the choice (walk-forward).
- **Final model:** fitted once on every training row, writing the Kaggle submission.

### Setup

In [1]:
# Standard library
import json
import sys
import warnings
from pathlib import Path

# Arrays, tables, the model and its metrics
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

# The group's own functions (src/3_Feature_selection/preprocessing.py and selection.py)
sys.path.append("../../../src/3_Feature_selection")
from preprocessing import (RANDOM_STATE, fold_matrices, load_data, paired_delta, purchase_dates,
                           target, temporal_splits)
from selection import make_strategy_selector

# Functions of the model notebooks (src/4_Model/model.py)
sys.path.append("../../../src/4_Model")
from model import (CUTOFFS, baseline_rule, best_cutoff, f1_by_cutoff, fit_with_cutoff,
                   score_configurations, walk_forward)

warnings.filterwarnings("ignore", category=UserWarning)  # unknown levels in validation, as in feature selection

In [2]:
train_table = pd.read_csv("../../../data/featureselection/feature_selection_l1.csv")
test_table = pd.read_csv("../../../data/featureselection/feature_selection_l1_test.csv")
features = [c for c in train_table.columns if c not in ("ID", "customer_id", target)]
assert list(test_table.columns) == list(train_table.columns.drop(target))

final_model, _, final_cutoff = fit_forest_with_cutoff(
    make_forest(**{**final_settings, "n_estimators": 1000}), train_table[features], train_table[target])
test_probabilities = final_model.predict_proba(test_table[features])[:, 1]
submission = pd.DataFrame({"ID": test_table["ID"], target: (test_probabilities >= final_cutoff).astype(int)})

sample = pd.read_csv("../../../data/sample_submission.csv")
assert list(submission["ID"]) == list(sample["ID"]) and set(submission[target]) <= {0, 1}

submission_file = Path("../../../data/submissions/ML_Group_18_Version02.csv")
submission_file.parent.mkdir(exist_ok=True)
submission.to_csv(submission_file, index=False)

previous = pd.read_csv("../../../data/submissions/ML_Group_18_Version01.csv")
agreement = (previous[target].to_numpy() == submission[target].to_numpy()).mean()
print(f"{len(features)} features | cut-off {final_cutoff} | {submission[target].mean():.1%} of the"
      f" test rows predicted to come back | saved to {submission_file.as_posix()}")
print(f"agreement with the logistic regression of version 01: {agreement:.1%}")

NameError: name 'fit_forest_with_cutoff' is not defined

In [ ]:
# The feature-selection strategy: the candidate columns and the selector's settings
strategy = json.loads(Path("../../../logs/feature_selection.json").read_text(encoding="utf-8"))
numeric = strategy["candidate_columns"]["numeric"]
categorical = strategy["candidate_columns"]["categorical"]
selector = strategy["selector"]

train_clientes = load_data()
# The splits need the raw date parts, so they are built before the calendar columns are
# replaced by their sin/cos components, as in the feature-selection notebook
time_splits = temporal_splits(train_clientes)
for col, period in [("purchase_month", 12.0), ("purchase_day", 31.0), ("purchase_dayofweek", 7.0)]:
    angle = 2.0 * np.pi * train_clientes[col].astype(float) / period
    train_clientes[f"{col}_sin"] = np.sin(angle)
    train_clientes[f"{col}_cos"] = np.cos(angle)
frame = train_clientes[numeric + categorical + [target]]

# The pre-processing team's rules (1.6, 1.7.3, 1.7.4 and 1.8), as in the baseline notebook
labels_as_numbers = ["is_first_purchase", "is_weekend", "since_2015_or_earlier", "purchase_year",
                     "customer_since_year", "purchase_month_sin", "purchase_month_cos",
                     "purchase_day_sin", "purchase_day_cos", "purchase_dayofweek_sin",
                     "purchase_dayofweek_cos"]
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": [(numeric, SimpleImputer(strategy="median")),
                   (categorical, SimpleImputer(strategy="most_frequent"))],
          "log1p": [c for c in numeric
                    if c not in labels_as_numbers and not (train_clientes[c] < 0).any()],
          "rare": {"transport_method": 50, "payment_type": 20, "city": 30, "municipality": 30,
                   "postal_area": 20}}
print(f"{len(frame):,} training rows | {len(numeric)} numeric and {len(categorical)} categorical"
      f" candidate columns | {len(time_splits)} temporal splits")

## 4.2 Assessment on the Temporal Splits

A random forest does not need scaled columns or a penalty to cope with correlated ones, so the feature selection is no longer a necessity. Two versions of the forest are assessed on the same splits, to see whether the selection still helps:

- **all columns:** every column the recipe produces;
- **selected columns:** the columns kept by the strategy of `logs/feature_selection.json` (multicollinearity filter at 0.90, then L1), fitted inside each split.

The first settings are a reasonable starting point, not tuned: 300 trees, `min_samples_leaf=5` (leaves of a single row would memorise the training rows) and `max_features="sqrt"`. The tuning of 4.5 searches around them.

In [ ]:
def make_forest(**settings):
    """A fresh, unfitted random forest. `oob_score=True` gives the out-of-bag probabilities that
    choose the cut-off; the defaults are the starting point of 4.2."""
    params = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": "sqrt"}
    params.update(settings)
    return RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=RANDOM_STATE, **params)


def assess(use_selection):
    """The 7 splits for one version of the forest: its measures per split and the matrices it used."""
    rows, pooled, prepared = [], [], []
    dates = purchase_dates(train_clientes)
    for train_rows, validation_rows in time_splits:
        train, validation = frame.iloc[train_rows], frame.iloc[validation_rows]
        A_train, A_validation, names = fold_matrices(train, validation, recipe)
        keep = np.ones(len(names), dtype=bool)
        if use_selection:
            keep = make_strategy_selector(selector).fit(A_train, train[target]).get_support()
        prepared.append((A_train[:, keep], A_validation[:, keep], train[target], validation[target], names[keep]))

        # The cut-off is chosen on the out-of-bag probabilities of the training rows
        model, p_oob, cutoff = fit_forest_with_cutoff(make_forest(), A_train[:, keep], train[target])
        p_validation = model.predict_proba(A_validation[:, keep])[:, 1]
        y, predicted = validation[target], (p_validation >= cutoff).astype(int)
        rows.append({"validation from": dates.iloc[validation_rows].min().date(),
                     "until": dates.iloc[validation_rows].max().date(),
                     "columns": int(keep.sum()), "cut-off": cutoff,
                     "OOB F1": f1_score(train[target], p_oob >= cutoff),
                     "F1": f1_score(y, predicted),
                     "AP": average_precision_score(y, p_validation),
                     "ROC-AUC": roc_auc_score(y, p_validation),
                     "balanced accuracy": balanced_accuracy_score(y, predicted),
                     "precision": precision_score(y, predicted),
                     "recall": recall_score(y, predicted)})
        pooled.append(pd.DataFrame({"actual": y.to_numpy(), "predicted": predicted}))
    return pd.DataFrame(rows), pooled, prepared


measures = ["OOB F1", "F1", "AP", "ROC-AUC", "balanced accuracy", "precision", "recall"]
runs = {"all columns": assess(use_selection=False), "selected columns": assess(use_selection=True)}
for name, (table, _, _) in runs.items():
    print(f"=== forest on {name}")
    print(table[["validation from", "until", "columns", "cut-off"] + measures].round(3).to_string(index=False))
    print("\nmean:", table[measures].mean().round(4).to_dict())
    print("SEM: ", (table[measures].std(ddof=1) / np.sqrt(len(table))).round(4).to_dict(), "\n")

## 4.3 Against the Logistic Regression and the Floors

The forest is compared, split by split, with the logistic regression of the baseline notebook (same recipe, same selection, cut-off chosen on the training rows), with the competition's baseline rule (`purchase_count_365d >= 3`) and with always predicting a repeat purchase. As in the other notebooks, **AP decides** and F1 is shown beside it; a difference counts only when it is larger than two standard errors.

In [ ]:
dates = purchase_dates(train_clientes)
logistic, floors = [], []
for (A_train, A_validation, y_train, y_validation, _), (_, validation_rows) in zip(runs["selected columns"][2], time_splits):
    model, _, cutoff = fit_with_cutoff(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), A_train, y_train)
    p = model.predict_proba(A_validation)[:, 1]
    logistic.append({"AP": average_precision_score(y_validation, p), "F1": f1_score(y_validation, p >= cutoff)})
    validation = frame.iloc[validation_rows]
    floors.append({"always 1": f1_score(y_validation, np.ones(len(y_validation), dtype=int)),
                   "baseline rule": f1_score(y_validation, baseline_rule(validation))})
logistic, floors = pd.DataFrame(logistic), pd.DataFrame(floors)

forest_all, forest_selected = runs["all columns"][0], runs["selected columns"][0]
summary = pd.DataFrame({
    "logistic regression": {"AP": logistic["AP"].mean(), "F1": logistic["F1"].mean()},
    "forest, all columns": {"AP": forest_all["AP"].mean(), "F1": forest_all["F1"].mean()},
    "forest, selected columns": {"AP": forest_selected["AP"].mean(), "F1": forest_selected["F1"].mean()},
    "baseline rule": {"AP": np.nan, "F1": floors["baseline rule"].mean()},
    "always 1": {"AP": np.nan, "F1": floors["always 1"].mean()}}).T
print(summary.round(4).to_string(), "\n")

def compare(label, a, b, metric):
    r = paired_delta(a, b)
    print(f"{label:55s} d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

for metric in ("AP", "F1"):
    compare("forest (selected) minus logistic regression", forest_selected[metric], logistic[metric], metric)
    compare("forest (all columns) minus logistic regression", forest_all[metric], logistic[metric], metric)
    compare("forest (all columns) minus forest (selected)", forest_all[metric], forest_selected[metric], metric)
compare("forest (selected) minus baseline rule", forest_selected["F1"], floors["baseline rule"], "F1")

## 4.4 Reading the Errors and the Importances

The confusion matrix gathers the predictions of the 7 validation blocks. The importance of each column is the mean decrease in impurity of a forest fitted on each split's training rows (selected columns), averaged over the splits where the column was kept.

In [ ]:
both = pd.concat(runs["selected columns"][1], ignore_index=True)
tn, fp, fn, tp = confusion_matrix(both["actual"], both["predicted"]).ravel()
print(f"{len(both):,} validation purchases: {tp} repeat purchases caught, {fn} missed,"
      f" {fp} false alarms, {tn} correctly left out\n")

importances = []
for A_train, _, y_train, _, names in runs["selected columns"][2]:
    importances.append(pd.Series(make_forest().fit(A_train, y_train).feature_importances_, index=names))
importances = pd.concat(importances, axis=1)
top = pd.DataFrame({"mean importance": importances.mean(axis=1),
                    "kept in": importances.notna().sum(axis=1).astype(str) + f"/{importances.shape[1]}"})
print(top.sort_values("mean importance", ascending=False).head(12).round(4).to_string())

## 4.5 Tuning the Random Forest

The forest has several settings that control how closely it fits the training rows. A random search tries combinations of:

- **`max_depth`:** none, 6, 10 or 15;
- **`min_samples_leaf`:** 1, 3, 5, 10 or 20;
- **`max_features`:** `"sqrt"`, `"log2"`, 0.3 or 0.5 of the columns at each split;
- **`class_weight`:** none, or `"balanced_subsample"`;
- **`max_samples`:** every row in each tree's bootstrap, or 70%.

The search runs on the same 7 splits and on the matrices of 4.2 (selected columns), plus the starting point of 4.2, with 150 trees each to keep the search affordable (the final model of 4.6 uses more). Each combination chooses its own cut-off on its out-of-bag probabilities. AP decides. As for the logistic regression, the search is then **scored on splits that took no part in the choice** (walk-forward) and is adopted only if it beats the starting point by more than two standard errors.

In [ ]:
space = {"max_depth": [None, 6, 10, 15], "min_samples_leaf": [1, 3, 5, 10, 20],
         "max_features": ["sqrt", "log2", 0.3, 0.5], "class_weight": [None, "balanced_subsample"],
         "max_samples": [None, 0.7]}
starting_point = {"max_depth": None, "min_samples_leaf": 5, "max_features": "sqrt",
                  "class_weight": None, "max_samples": None}
sampled = list(ParameterSampler(space, n_iter=10, random_state=RANDOM_STATE))
grid = [starting_point] + [c for c in sampled if c != starting_point]
default = 0


def config_name(index):
    return ", ".join(f"{key}={value}" for key, value in grid[index].items())


rows = []
for index, config in enumerate(grid):
    for split, (A_train, A_validation, y_train, y_validation, _) in enumerate(runs["selected columns"][2]):
        model, _, cutoff = fit_forest_with_cutoff(make_forest(**{**config, "n_estimators": 150}), A_train, y_train)
        p = model.predict_proba(A_validation)[:, 1]
        rows.append({"configuration": index, "split": split, "cut-off": cutoff,
                     "AP": average_precision_score(y_validation, p), "F1": f1_score(y_validation, p >= cutoff)})
scores = pd.DataFrame(rows)
mean_ap = scores.groupby("configuration")["AP"].mean()

reference_run = scores[scores["configuration"] == default].set_index("split")
board = {}
for index in dict.fromkeys(list(mean_ap.nlargest(5).index) + [default]):
    run = scores[scores["configuration"] == index].set_index("split")
    r = paired_delta(run["AP"], reference_run["AP"]) if index != default else {"delta": 0.0, "SEM": 0.0, "wins": "0/7", "verdict": "-"}
    board[config_name(index) + (" (starting point)" if index == default else "")] = {
        "AP": run["AP"].mean(), "dAP": r["delta"], "AP better on": r["wins"], "AP verdict": r["verdict"],
        "F1": run["F1"].mean(), "dF1": run["F1"].mean() - reference_run["F1"].mean()}
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
print(pd.DataFrame.from_dict(board, orient="index").round(4).to_string())

In [ ]:
# For each split from the third on: the combination chosen on the earlier splits, scored on it
honest = walk_forward(scores, first=2)
on_same = reference_run.loc[honest["split"]]
dates_of_splits = [dates.iloc[rows_[1]].min().date() for rows_ in time_splits]
print(pd.DataFrame({
    "validation from": [dates_of_splits[s] for s in honest["split"]],
    "chosen on the earlier splits": [config_name(i) for i in honest["chosen"]],
    "AP": honest["AP"].to_numpy(), "AP start": on_same["AP"].to_numpy(),
    "F1": honest["F1"].to_numpy(), "F1 start": on_same["F1"].to_numpy()}).round(4).to_string(index=False))

search_vs_start = pd.DataFrame({m: paired_delta(honest[m].to_numpy(), on_same[m].to_numpy())
                                for m in ("AP", "F1")}).T
print("\nsearch minus starting point on the splits it did not see:\n", search_vs_start.to_string())

chosen = int(mean_ap.idxmax()) if search_vs_start.loc["AP", "verdict"] == "better" else default
final_settings = grid[chosen]
print(f"\nbest combination on all 7 splits: {config_name(int(mean_ap.idxmax()))}")
print(f"settings of the final model: {config_name(chosen)}")

## 4.6 Final Model and Kaggle Submission

As in the baseline notebook, the final model repeats the steps of each split once, on every training row. `feature_selection_l1.csv` carries the training rows prepared by the recipe and reduced to the selected columns, and `feature_selection_l1_test.csv` the test rows prepared with the same fits. The forest has the settings chosen in 4.5 and, this time, **1000 trees**: a final model has no reason to keep the 300 used to compare combinations, and more trees only make the probabilities steadier. The cut-off is chosen on the out-of-bag probabilities of the training rows, and the submission keeps `ID` and writes 0/1 labels. This is version 02.

In [ ]:
train_table = pd.read_csv("../../data/featureselection/feature_selection_l1.csv")
test_table = pd.read_csv("../../data/featureselection/feature_selection_l1_test.csv")
features = [c for c in train_table.columns if c not in ("ID", "customer_id", target)]
assert list(test_table.columns) == list(train_table.columns.drop(target))

final_model, _, final_cutoff = fit_forest_with_cutoff(
    make_forest(**{**final_settings, "n_estimators": 1000}), train_table[features], train_table[target])
test_probabilities = final_model.predict_proba(test_table[features])[:, 1]
submission = pd.DataFrame({"ID": test_table["ID"], target: (test_probabilities >= final_cutoff).astype(int)})

sample = pd.read_csv("../../data/sample_submission.csv")
assert list(submission["ID"]) == list(sample["ID"]) and set(submission[target]) <= {0, 1}

submission_file = Path("../../data/submissions/ML_Group_18_Version02.csv")
submission_file.parent.mkdir(exist_ok=True)
submission.to_csv(submission_file, index=False)

previous = pd.read_csv("../../data/submissions/ML_Group_18_Version01.csv")
agreement = (previous[target].to_numpy() == submission[target].to_numpy()).mean()
print(f"{len(features)} features | cut-off {final_cutoff} | {submission[target].mean():.1%} of the"
      f" test rows predicted to come back | saved to {submission_file.as_posix()}")
print(f"agreement with the logistic regression of version 01: {agreement:.1%}")